In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np
import cv2
from sklearn.utils import shuffle
from tensorflow.keras.utils import to_categorical
import shutil
from tensorflow.keras import layers
from tensorflow.keras.models import Model
import random
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing import image

In [ ]:
def carregar_dados(caminho_train, caminho_val):

    # Inicialize as listas de dados
    x_train = []
    y_train = []
    x_test = []
    y_test = []

    # Função para redimensionar as imagens
    def redimensionar_imagem(imagem, dimensao=(128, 128, 3)):
        imagem_redimensionada = cv2.resize(imagem, (dimensao[1], dimensao[0]))
        return imagem_redimensionada

    # Função para carregar as imagens de uma pasta
    def ler_imagens(pasta, rotulo):
        for arquivo in os.listdir(pasta):
            if arquivo.lower().endswith('.jpg'):
                caminho_arquivo = os.path.join(pasta, arquivo)
                imagem = cv2.imread(caminho_arquivo)

                # Adicione qualquer pré-processamento necessário aqui
                imagem_redimensionada = redimensionar_imagem(imagem)

                if pasta.startswith(caminho_train):
                    x_train.append(imagem_redimensionada)
                    y_train.append(rotulo)
                elif pasta.startswith(caminho_val):
                    x_test.append(imagem_redimensionada)
                    y_test.append(rotulo)

    # Percorra as pastas em train
    for rotulo, pasta in enumerate(os.listdir(caminho_train)):
        caminho_pasta = os.path.join(caminho_train, pasta)
        if os.path.isdir(caminho_pasta):
            ler_imagens(caminho_pasta, rotulo)

    # Percorra as pastas em val
    for rotulo, pasta in enumerate(os.listdir(caminho_val)):
        caminho_pasta = os.path.join(caminho_val, pasta)
        if os.path.isdir(caminho_pasta):
            ler_imagens(caminho_pasta, rotulo)

    # Converta as listas embaralhadas para arrays numpy
    x_train = np.array(x_train)
    y_train = np.array(y_train)
    x_test = np.array(x_test)
    y_test = np.array(y_test)

    # Converta os rótulos para valores inteiros
    y_train = y_train.astype('int32')
    y_test = y_test.astype('int32')

    # Aplique a normalização
    x_train = x_train / 255.0
    x_test = x_test / 255.0

    return x_train, y_train, x_test, y_test

In [ ]:
caminho_train = '/content/drive/MyDrive/train'
caminho_val =  '/content/drive/MyDrive/val'
x_train, y_train, x_test, y_test = carregar_dados(caminho_train, caminho_val)

In [ ]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)
y_test = encoder.fit_transform(y_test)

In [ ]:
input = layers.Input(shape=(128, 128, 3))

# Encoder
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(input)
x = layers.MaxPooling2D((2, 2), padding="same")(x)
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2), padding="same")(x)

# Decoder
x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation="relu", padding="same")(x)
x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation="relu", padding="same")(x)
x = layers.Conv2D(3, (3, 3), activation="softmax", padding="same")(x)

# Autoencoder
autoencoder = Model(input, x)
autoencoder.compile(optimizer="adam", loss="binary_crossentropy",  metrics=["accuracy"])
autoencoder.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128, 128, 3)]     0         
                                                                 
 conv2d (Conv2D)             (None, 128, 128, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 64, 64, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 64, 64, 32)        9248      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 32, 32, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_transpose (Conv2DTr  (None, 64, 64, 32)        9248  

In [ ]:
autoencoder.fit(
    x=x_train,
    y=x_train,
    epochs=100,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test, x_test),
)